# Site API access points and live tests

Use two coordinates to retrieve public Massachusetts site evidence. This notebook tests the API routes, queries all eight automated providers individually, consumes their records with Polars, and checks expected errors. Saved outputs below come from real HTTP requests.

Start the backend in a separate terminal from the repository root:

```sh
uv pip install -r requirements.txt
uv run --env-file .env uvicorn API.server:app --reload --port 8001
```

Then open this notebook with `uv run --env-file .env jupyter lab site_api_demo.ipynb` and choose **Run All**. You can omit `--env-file .env` if you have no `.env` file. OpenEI then uses its public demo key. The first solar lookup may download a 13 MB workbook.

Edit `BASE_URL`, `LATITUDE`, and `LONGITUDE` below. The default Cambridge point intersects an MIT campus parcel whose assessor address is 60 Vassar St. Parcel, building and meter boundaries still need confirmation. See [the API guide](docs/site_api.md) for matching methods and limits.

OpenEI's shared `DEMO_KEY` can return HTTP 429. For a personal key, set `OPENEI_API_KEY` in the server's Git-ignored `.env`, restart the API process, and rerun the tariff and summary cells. If the combined lookup was partial, rerun that cell too. The notebook always reports provider failures. Set `REQUIRE_ALL_PROVIDERS=True` to make them stop the final test cell with an assertion.

In [1]:
import json
from pathlib import Path
from time import perf_counter

import polars as pl
import requests
from IPython.display import Markdown, display

BASE_URL = "http://127.0.0.1:8001"
LATITUDE, LONGITUDE = 42.359, -71.093
REQUIRE_ALL_PROVIDERS = False  # True makes any provider failure fail the final cell.
POINT = {"latitude": LATITUDE, "longitude": LONGITUDE}
OUTPUT_DIR = Path("data/processed/api_demo")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

session = requests.Session()
http_tests = []
source_results = {}

# One request helper keeps every test's URL, HTTP status and duration visible.
def get_endpoint(name, path, params=None, expected_status=200):
    started = perf_counter()
    response = session.get(f"{BASE_URL}{path}", params=params, timeout=120)
    passed = response.status_code == expected_status
    http_tests.append({
        "test": name, "method": "GET", "url": response.url,
        "expected_status": expected_status, "actual_status": response.status_code,
        "result": "PASS" if passed else "FAIL",
        "seconds": round(perf_counter() - started, 3),
    })
    print(f"{name}: HTTP {response.status_code} ({http_tests[-1]['seconds']:.3f}s)")
    assert passed, f"Expected HTTP {expected_status}. Response: {response.text[:500]}"
    return response

## 1. GET /health

This checks that the server is reachable without calling a provider. A connection error means you should start the server or correct `BASE_URL` before proceeding.

In [2]:
health = get_endpoint("Health", "/health")
assert health.json() == {"status": "ok"}
display(health.json())

Health: HTTP 200 (0.005s)


{'status': 'ok'}

## 2. GET /sources

The catalog provides provider IDs, access URLs, evidence types and matching methods. `references` distinguishes manual lookups, customer authorization and explicit model selection from automatically retrieved data.

In [3]:
catalog = get_endpoint("Source catalog", "/sources").json()
assert len(catalog["automated"]) == 8
display(pl.from_dicts([
    {"source": name, "url": item["url"], "data_kind": item["data_kind"],
     "match_method": item["match_method"], "description": item["description"]}
    for name, item in catalog["automated"].items()
]))
display(pl.from_dicts([
    {"source": name, "access": item["status"], "url": item.get("url"), "action": item["description"]}
    for name, item in catalog["references"].items()
]))

Source catalog: HTTP 200 (0.002s)


source,url,data_kind,match_method,description
str,str,str,str,str
"""utilities""","""https://arcgisserver.digital.m…","""published_utility_territory""","""municipality_containing_point""","""Municipality and its published…"
"""parcels""","""https://services1.arcgis.com/h…","""reported_assessor""","""point_intersection""","""Containing assessor parcels, o…"
"""buildings""","""https://services1.arcgis.com/h…","""published_geometry""","""point_intersection""","""Mapped roof/building outline i…"
"""lber""","""https://services1.arcgis.com/7…","""reported""","""within_250m_candidate""","""Up to five 2025 annual disclos…"
"""flood""","""https://hazards.fema.gov/arcgi…","""published_hazard""","""point_intersection""","""FEMA flood-zone polygons at th…"
"""tariffs""","""https://api.openei.org/utility…","""published_tariff""","""provider_location_candidates""","""Commercial tariff candidates n…"
"""solar""","""https://www.masscec.com/sites/…","""reported_municipal_context""","""municipality_only""","""January 2025 PTS municipal sol…"
"""elevation""","""https://api.open-meteo.com/v1/…","""modeled_terrain""","""coordinate_sample""","""Terrain elevation in meters. I…"


source,access,url,action
str,str,str,str
"""land_records""","""manual_lookup""","""https://www.masslandrecords.co…","""Search the appropriate registr…"
"""permits""","""manual_lookup""",null,"""Municipal portals vary. Search…"
"""tariff_documents""","""reference_only""","""https://www.mass.gov/info-deta…","""Official tariff directory. Mat…"
"""requirements""","""reference_only""","""https://www.mass.gov/regulatio…","""Identify current building, fir…"
"""bills""","""authorization_required""","""https://www.eqore.com/guides/i…","""Obtain twelve bills for each r…"
"""interval_data""","""authorization_required""","""https://www.eqore.com/guides/i…","""Obtain customer-authorized met…"
"""comstock""","""selection_required""","""https://natlabrockies.github.i…","""Select an appropriate modeled …"


## 3. GET /openapi.json, /docs and /redoc

`/openapi.json` describes the callable API and its query parameters. `/docs` and `/redoc` provide interactive/reference pages. A successful HTML test checks the served page; rendering these pages in a browser also requires their frontend assets.

In [4]:
schema = get_endpoint("OpenAPI schema", "/openapi.json").json()
assert {"/health", "/sources", "/site"} <= set(schema["paths"])
parameters = schema["paths"]["/site"]["get"]["parameters"]
assert {"latitude", "longitude", "sources", "refresh"} <= {p["name"] for p in parameters}
display(pl.from_dicts([
    {"parameter": p["name"], "required": p.get("required", False), "description": p.get("description", "")}
    for p in parameters
]))

docs = get_endpoint("Swagger documentation", "/docs")
assert "text/html" in docs.headers["content-type"]
redoc = get_endpoint("ReDoc documentation", "/redoc")
assert "text/html" in redoc.headers["content-type"]
display(Markdown(f"[Open Swagger UI]({BASE_URL}/docs) | [Open ReDoc]({BASE_URL}/redoc)"))

OpenAPI schema: HTTP 200 (0.006s)


parameter,required,description
str,bool,str
"""latitude""",true,"""WGS84 latitude. Choose a point…"
"""longitude""",true,"""WGS84 longitude."""
"""sources""",false,"""Optional comma-separated autom…"
"""refresh""",false,"""Download a fresh MassCEC workb…"


Swagger documentation: HTTP 200 (0.001s)
ReDoc documentation: HTTP 200 (0.001s)


[Open Swagger UI](http://127.0.0.1:8001/docs) | [Open ReDoc](http://127.0.0.1:8001/redoc)

## 4. GET /site with latitude and longitude

This is the normal frontend/backend call. The response contains all twelve checklist categories, provider records, provenance, manual references and missing inputs. `status=ok` means automatic calls completed. It does not mean the site is ready for engineering.

In [5]:
site = get_endpoint("Full coordinate lookup", "/site", POINT).json()
assert site["schema_version"] == "1.0"
assert len(site["checklist"]) == 12
assert set(site["sources"]) == set(catalog["automated"])
(OUTPUT_DIR / "site_lookup.json").write_text(json.dumps(site, indent=2))
print("Location:", site["location"])
print("Overall retrieval status:", site["status"])
print("Provider errors:", site["source_errors"])
assert site["status"] == ("partial" if site["source_errors"] else "ok")
display(pl.from_dicts([
    {"source": name, "status": item["status"], "rows": len(item["records"]),
     "seconds": item["elapsed_seconds"], "error": item.get("error", "")}
    for name, item in site["sources"].items()
]))

Full coordinate lookup: HTTP 200 (1.215s)
Location: {'latitude': 42.359, 'longitude': -71.093, 'municipality': 'CAMBRIDGE', 'state': 'MA', 'address': '60 VASSAR ST'}
Overall retrieval status: ok
Provider errors: []


source,status,rows,seconds,error
str,str,i64,f64,str
"""utilities""","""ok""",1,0.147,""""""
"""parcels""","""ok""",1,0.376,""""""
"""buildings""","""ok""",1,0.241,""""""
"""lber""","""ok""",5,0.256,""""""
"""flood""","""ok""",1,0.981,""""""
"""tariffs""","""ok""",2,0.459,""""""
"""solar""","""ok""",5,1.06,""""""
"""elevation""","""ok""",1,0.756,""""""


## 5. Test each provider separately

The same endpoint accepts `sources=<provider ID>` for debugging. Utilities is always queried to resolve Massachusetts coverage and municipality. Each cell below makes a separate live request and displays a small table of the returned data.

`no_data` is a valid empty lookup, distinct from an upstream error. An empty result does not establish absence of equipment, reporting obligations or hazards. Provider failures appear as FAIL in the final summary. Strict mode also raises an assertion.

### utilities: Municipality and utility providers

A municipality-level provider list is a candidate. Confirm the utility supplying the actual account from a bill.

In [6]:
payload = get_endpoint("Provider: utilities", "/site", {**POINT, "sources": "utilities"}).json()
result = payload["sources"]["utilities"]
source_results["utilities"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['TOWN', 'TOWN_ID', 'ELEC']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['TOWN', 'TOWN_ID', 'ELEC']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: utilities: HTTP 200 (0.144s)
Provider status: ok | Records: 1
Source: https://arcgisserver.digital.mass.gov/arcgisserver/rest/services/AGOL/ElectricityProviders/MapServer/0
Matching: municipality_containing_point | Retrieved: 2026-11-07T18:39:14.194895+00:00


TOWN,TOWN_ID,ELEC
str,i64,str
"""CAMBRIDGE""",49,"""Eversource Energy (NSTAR Elect…"


### parcels: Assessor parcel records

Inspect the containing parcel, reported owner, assessor area, zoning and deed references. Multiple intersections remain ambiguous. Geometries are retained in the JSON but omitted from this table.

In [7]:
payload = get_endpoint("Provider: parcels", "/site", {**POINT, "sources": "parcels"}).json()
result = payload["sources"]["parcels"]
source_results["parcels"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['LOC_ID', 'SITE_ADDR', 'CITY', 'OWNER1', 'BLD_AREA', 'FY', 'ZONING', 'LS_BOOK', 'LS_PAGE']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['LOC_ID', 'SITE_ADDR', 'CITY', 'OWNER1', 'BLD_AREA', 'FY', 'ZONING', 'LS_BOOK', 'LS_PAGE']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: parcels: HTTP 200 (3.581s)
Provider status: ok | Records: 1
Source: https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0
Matching: point_intersection | Retrieved: 2026-11-07T18:39:17.784055+00:00


LOC_ID,SITE_ADDR,CITY,OWNER1,BLD_AREA,FY,ZONING,LS_BOOK,LS_PAGE
str,str,str,str,i64,i64,str,str,str
"""F_766725_2956445""","""60 VASSAR ST""","""CAMBRIDGE""","""MASSACHUSETTS INSTITUTE OF TEC…",56880,2026,"""C-3""","""00906""","""0350"""


### buildings: Mapped building and roof outlines

The point must intersect a mapped outline. Roof area is distinct from gross floor area and usable battery space. Review source dates before treating geometry as current.

In [8]:
payload = get_endpoint("Provider: buildings", "/site", {**POINT, "sources": "buildings"}).json()
result = payload["sources"]["buildings"]
source_results["buildings"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['STRUCT_ID', 'AREA_SQ_FT', 'SOURCE', 'SOURCETYPE', 'SOURCEDATE']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['STRUCT_ID', 'AREA_SQ_FT', 'SOURCE', 'SOURCETYPE', 'SOURCEDATE']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: buildings: HTTP 200 (2.147s)
Provider status: ok | Records: 1
Source: https://services1.arcgis.com/hGdibHYSPO59RG1h/ArcGIS/rest/services/Building_Structures/FeatureServer/0
Matching: point_intersection | Retrieved: 2026-11-07T18:39:19.940667+00:00


STRUCT_ID,AREA_SQ_FT,SOURCE,SOURCETYPE,SOURCEDATE
str,f64,str,str,i64
"""233661_901110""",619467.03567,"""ROLTA""","""ROOFPRINT""",20110000


### lber: Nearby annual energy disclosures

These are up to five energy-year 2025 candidates within 250 m. They are not automatically assigned to the facility or meter. Purchased-grid electricity is normalized from kBtu to kWh alongside original fields.

In [9]:
payload = get_endpoint("Provider: lber", "/site", {**POINT, "sources": "lber"}).json()
result = payload["sources"]["lber"]
source_results["lber"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['building_id', 'address', 'city', 'data_year', 'floor_area_sqft', 'annual_electricity_kwh']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['building_id', 'address', 'city', 'data_year', 'floor_area_sqft', 'annual_electricity_kwh']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: lber: HTTP 200 (1.931s)
Provider status: ok | Records: 5
Source: https://services1.arcgis.com/7iJyYTjCtKsZS1LR/arcgis/rest/services/2026_10_28_LBER_Annual_Report_Data/FeatureServer/0
Matching: within_250m_candidate | Retrieved: 2026-11-07T18:39:21.882940+00:00


building_id,address,city,data_year,floor_area_sqft,annual_electricity_kwh
str,str,str,i64,f64,f64
"""87JC9W54+J39-14-18-13-18""","""84 Massachusetts Ave""","""Cambridge""",2025,370860.0,2.3243e6
"""87JC9W54+3J5-15-15-16-16""","""305 Memorial Dr""","""Cambridge""",2025,237928.0,2.6438e6
"""87JC9W65+2FJ-82-97-95-99""","""60 Vassar St""","""Cambridge""",2025,3.78852e6,9.5897e6
"""87JC9W53+RX6-16-17-15-19""","""201 Amherst St""","""Cambridge""",2025,140738.2,738843.304633
"""87JC9W53+7X4-10-9-7-10""","""48-R Massachusetts Ave""","""Cambridge""",2025,58699.17,381735.03333


### tariffs: Candidate commercial tariffs

These are nearby candidate rates and can be historical or inapplicable. Review utility, eligibility and effective dates. The full nested rate definition is available in each record’s `rate` object.

In [10]:
payload = get_endpoint("Provider: tariffs", "/site", {**POINT, "sources": "tariffs"}).json()
result = payload["sources"]["tariffs"]
source_results["tariffs"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['label', 'utility', 'name', 'startdate', 'enddate']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['label', 'utility', 'name', 'startdate', 'enddate']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: tariffs: HTTP 200 (0.522s)
Provider status: ok | Records: 2
Source: https://api.openei.org/utility_rates
Matching: provider_location_candidates | Retrieved: 2026-11-07T18:39:22.416739+00:00


label,utility,name,startdate,enddate
str,str,str,i64,i64
"""539f6b90ec4f024411ec9e1f""","""NSTAR Electric Company""","""Cambridge Electric Light Compa…",1396360800,1427788800
"""539f6becec4f024411eca345""","""NSTAR Electric Company""","""Cambridge Electric Light Compa…",1396360800,1427788800


In [11]:
if source_results["tariffs"]["records"]:
    rate = source_results["tariffs"]["records"][0]["rate"]
    print("Available nested rate fields:", sorted(rate))
    print("Demand tiers:", json.dumps(rate.get("demandratestructure"), indent=2))

Available nested rate fields: ['approved', 'country', 'demandrateunit', 'demandunits', 'description', 'eiaid', 'enddate', 'energycomments', 'energyratestructure', 'energyweekdayschedule', 'energyweekendschedule', 'fixedchargefirstmeter', 'fixedchargeunits', 'flatdemandmonths', 'flatdemandstructure', 'flatdemandunit', 'label', 'name', 'peakkwcapacitymax', 'peakkwcapacitymin', 'phasewiring', 'revisions', 'sector', 'source', 'sourceparent', 'startdate', 'uri', 'utility', 'voltagecategory']
Demand tiers: null


### solar: Municipal solar context

The January 2025 MassCEC workbook has no street addresses or coordinates. The API returns a sample and total count for the municipality. These records do not establish whether this facility has solar.

In [12]:
payload = get_endpoint("Provider: solar", "/site", {**POINT, "sources": "solar"}).json()
result = payload["sources"]["solar"]
source_results["solar"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['City', 'Capacity \n(DC, kW)', 'Facility Type', 'Utility']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['City', 'Capacity \n(DC, kW)', 'Facility Type', 'Utility']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: solar: HTTP 200 (1.842s)
Provider status: ok | Records: 5
Source: https://www.masscec.com/sites/default/files/documents/Solar-PV-Systems-in-MA-as-of-Jan-2025.xlsx
Matching: municipality_only | Retrieved: 2026-11-07T17:25:43.566150+00:00


City,"Capacity (DC, kW)",Facility Type,Utility
str,f64,str,str
"""Cambridge""",33.3,"""Multi-family residential (4 or…","""NSTAR (DBA EverSource)"""
"""Cambridge""",26.19,"""Multi-family residential (4 or…","""NSTAR (DBA EverSource)"""
"""Cambridge""",33.465,"""Multi-family residential (4 or…","""NSTAR (DBA EverSource)"""
"""Cambridge""",15.3,"""Multi-family residential (4 or…","""NSTAR (DBA EverSource)"""
"""Cambridge""",32.16,"""College / University""","""NSTAR (DBA EverSource)"""


In [13]:
print("Total municipal solar records:", source_results["solar"].get("total_municipal_records"))
print("Workbook data date:", source_results["solar"].get("data_as_of"))

Total municipal solar records: 881
Workbook data date: 2025-01-14


### flood: FEMA flood-zone evidence

These attributes describe flood polygons at the requested point. They do not assess every part of the parcel. No returned feature leaves flood risk unresolved.

In [14]:
payload = get_endpoint("Provider: flood", "/site", {**POINT, "sources": "flood"}).json()
result = payload["sources"]["flood"]
source_results["flood"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['FLD_ZONE', 'ZONE_SUBTY', 'SFHA_TF', 'DFIRM_ID']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['FLD_ZONE', 'ZONE_SUBTY', 'SFHA_TF', 'DFIRM_ID']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: flood: HTTP 200 (0.978s)
Provider status: ok | Records: 1
Source: https://hazards.fema.gov/arcgis/rest/services/public/NFHL/MapServer/28
Matching: point_intersection | Retrieved: 2026-11-07T18:39:25.275917+00:00


FLD_ZONE,ZONE_SUBTY,SFHA_TF,DFIRM_ID
str,str,str,str
"""X""","""AREA OF MINIMAL FLOOD HAZARD""","""F""","""25017C"""


### elevation: Terrain elevation

The returned value is terrain elevation in meters. It does not establish finished floor or pad elevation, structural suitability or subsurface conditions.

In [15]:
payload = get_endpoint("Provider: elevation", "/site", {**POINT, "sources": "elevation"}).json()
result = payload["sources"]["elevation"]
source_results["elevation"] = result
print("Provider status:", result["status"], "| Records:", len(result["records"]))
print("Source:", result["source_url"])
print("Matching:", result["match_method"], "| Retrieved:", result["retrieved_at_utc"])
if result["records"]:
    assert set(['elevation_m']) <= set(result["records"][0]), "Provider fields changed. Inspect its records before consuming them."
    sample = pl.from_dicts([{column: row.get(column) for column in ['elevation_m']} for row in result["records"]], infer_schema_length=None)
    display(sample)
else:
    print(result.get("error", result.get("detail", "No published records returned.")))

Provider: elevation: HTTP 200 (0.247s)
Provider status: ok | Records: 1
Source: https://api.open-meteo.com/v1/elevation
Matching: coordinate_sample | Retrieved: 2026-11-07T18:39:25.531140+00:00


elevation_m
f64
17.0


## 6. Four-column checklist table

Format the already-fetched response into `checklist item`, `subcategory_name`, `subcategory_value`, and `data_source`. This does not make additional provider calls. Values are text or null. Candidate names preserve their scope, and source errors get separate `*.error` rows. Use [site_checklist.ipynb](site_checklist.ipynb) for the shorter latitude/longitude-to-table notebook.

In [16]:
from API import checklist_table

table = checklist_table(site)
with pl.Config(tbl_rows=-1, fmt_str_lengths=500):
    display(table)
table.write_parquet(OUTPUT_DIR / "checklist.parquet")
table.write_csv(OUTPUT_DIR / "checklist.csv")
missing = pl.from_dicts(site["missing_fields"])
missing.write_csv(OUTPUT_DIR / "missing_inputs.csv")
print(f"{missing.height} checklist fields remain unresolved.")

checklist item,subcategory_name,subcategory_value,data_source
str,str,str,str
"""Site identity and boundaries""","""address""","""60 VASSAR ST""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Site identity and boundaries""","""municipality""","""CAMBRIDGE""","""https://arcgisserver.digital.mass.gov/arcgisserver/rest/services/AGOL/ElectricityProviders/MapServer/0"""
"""Site identity and boundaries""","""state""","""MA""","""https://arcgisserver.digital.mass.gov/arcgisserver/rest/services/AGOL/ElectricityProviders/MapServer/0"""
"""Site identity and boundaries""","""latitude""","""42.359""","""User-provided coordinates"""
"""Site identity and boundaries""","""longitude""","""-71.093""","""User-provided coordinates"""
"""Site identity and boundaries""","""parcel_id""","""F_766725_2956445""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Site identity and boundaries""","""building_use""","""College or University (Educational Private)""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Site identity and boundaries""","""assessor_building_area_sqft""","""56880""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""
"""Contacts and authority""","""recorded_owner""","""MASSACHUSETTS INSTITUTE OF TECHNOLOGY""","""https://services1.arcgis.com/hGdibHYSPO59RG1h/arcgis/rest/services/Massachusetts_Property_Tax_Parcels/FeatureServer/0"""


38 checklist fields remain unresolved.


## 7. Check the same contract at another facility

Use a public LBER example in Fall River. Fetch a small source selection and verify that category IDs and field names remain consistent across sites. Sources omitted from the request become `not_requested` in the checklist.

In [17]:
second_site = get_endpoint(
    "Second facility and source filtering", "/site",
    {"latitude": 41.762125, "longitude": -71.100125, "sources": "parcels,buildings,lber"},
).json()
assert set(second_site["sources"]) == {"utilities", "parcels", "buildings", "lber"}
assert [c["id"] for c in second_site["checklist"]] == [c["id"] for c in site["checklist"]]
assert [[f["name"] for f in c["fields"]] for c in second_site["checklist"]] == [[f["name"] for f in c["fields"]] for c in site["checklist"]]
second_fields = {f["name"]: f for c in second_site["checklist"] for f in c["fields"]}
assert second_fields["flood_zone"]["status"] == "not_requested"
print(second_site["location"])
print("Same twelve categories and field names. Unrequested flood data stays not_requested.")

Second facility and source filtering: HTTP 200 (3.638s)
{'latitude': 41.762125, 'longitude': -71.100125, 'municipality': 'FALL RIVER', 'state': 'MA', 'address': '1180 INNOVATION WAY'}
Same twelve categories and field names. Unrequested flood data stays not_requested.


## 8. Test expected HTTP 422 responses

These are intentional negative tests. Missing or invalid parameters and unsupported source names are rejected before provider calls. A point outside Massachusetts municipal land coverage is rejected after the coverage query.

In [18]:
invalid_requests = [
    ("Missing latitude", {"longitude": LONGITUDE}),
    ("Invalid latitude", {"latitude": 91, "longitude": LONGITUDE}),
    ("Unknown source", {**POINT, "sources": "not-a-source"}),
    ("Outside Massachusetts", {"latitude": 40.7, "longitude": -74.0, "sources": "parcels"}),
]
errors = []
for name, params in invalid_requests:
    response = get_endpoint(name, "/site", params, expected_status=422)
    errors.append({"test": name, "http_status": response.status_code, "detail": json.dumps(response.json()["detail"])})
display(pl.from_dicts(errors))

Missing latitude: HTTP 422 (0.005s)
Invalid latitude: HTTP 422 (0.003s)
Unknown source: HTTP 422 (0.002s)
Outside Massachusetts: HTTP 422 (0.133s)


test,http_status,detail
str,i64,str
"""Missing latitude""",422,"""[{""type"": ""missing"", ""loc"": [""…"
"""Invalid latitude""",422,"""[{""type"": ""less_than_equal"", ""…"
"""Unknown source""",422,"""""Unknown sources: not-a-source…"
"""Outside Massachusetts""",422,"""""No Massachusetts municipality…"


## 9. Final test summary

The HTTP table checks every tested access point and expected error response. The provider table separately checks availability, required provenance and record counts. An HTTP 200 partial response can contain a failed provider. `no_data` is displayed explicitly and is accepted as a valid empty query.

Provider failures remain FAIL even when the notebook continues to show the usable data. Set `REQUIRE_ALL_PROVIDERS=True` for a strict automated run. A failure in the combined lookup is reported separately, because a later individual retry can succeed without updating that earlier response.

All tables and the full example response are saved under Git-ignored `data/processed/api_demo/`. No API keys are written to the notebook outputs.

In [19]:
http_summary = pl.from_dicts(http_tests)
provider_summary = pl.from_dicts([
    {"source": name, "status": result["status"], "records": len(result["records"]),
     "result": "PASS" if result["status"] in ("ok", "no_data") else "FAIL",
     "seconds": result["elapsed_seconds"], "match_method": result["match_method"],
     "retrieved_at_utc": result["retrieved_at_utc"], "error": result.get("error", "")}
    for name, result in source_results.items()
])
display(http_summary)
display(provider_summary)
http_summary.write_csv(OUTPUT_DIR / "http_tests.csv")
provider_summary.write_parquet(OUTPUT_DIR / "provider_tests.parquet")
assert http_summary["result"].eq("PASS").all()
assert set(source_results) == set(catalog["automated"])
successful = provider_summary.filter(pl.col("result") == "PASS")
failed = provider_summary.filter(pl.col("result") == "FAIL")
assert successful["retrieved_at_utc"].is_not_null().all()
assert all(result["source_url"] and result["match_method"] for result in source_results.values())
print(f"HTTP checks: {http_summary.height}/{http_summary.height} PASS.")
print(f"Individual provider checks: {successful.height}/{provider_summary.height} PASS, {failed.height} FAIL.")
print("Combined lookup provider errors:", site["source_errors"])
if failed.height:
    display(failed.select("source", "error"))
    print("For OpenEI HTTP 429, configure a personal OPENEI_API_KEY in .env, restart the server and retry.")
if REQUIRE_ALL_PROVIDERS:
    assert failed.is_empty(), "A provider failed. Inspect its error and retry its cell."
    assert not site["source_errors"], "The combined lookup had provider errors. Inspect and rerun it."

test,method,url,expected_status,actual_status,result,seconds
str,str,str,i64,i64,str,f64
"""Health""","""GET""","""http://127.0.0.1:8001/health""",200,200,"""PASS""",0.005
"""Source catalog""","""GET""","""http://127.0.0.1:8001/sources""",200,200,"""PASS""",0.002
"""OpenAPI schema""","""GET""","""http://127.0.0.1:8001/openapi.…",200,200,"""PASS""",0.006
"""Swagger documentation""","""GET""","""http://127.0.0.1:8001/docs""",200,200,"""PASS""",0.001
"""ReDoc documentation""","""GET""","""http://127.0.0.1:8001/redoc""",200,200,"""PASS""",0.001
…,…,…,…,…,…,…
"""Second facility and source fil…","""GET""","""http://127.0.0.1:8001/site?lat…",200,200,"""PASS""",3.638
"""Missing latitude""","""GET""","""http://127.0.0.1:8001/site?lon…",422,422,"""PASS""",0.005
"""Invalid latitude""","""GET""","""http://127.0.0.1:8001/site?lat…",422,422,"""PASS""",0.003


source,status,records,result,seconds,match_method,retrieved_at_utc,error
str,str,i64,str,f64,str,str,str
"""utilities""","""ok""",1,"""PASS""",0.142,"""municipality_containing_point""","""2026-11-07T18:39:14.194895+00:…",""""""
"""parcels""","""ok""",1,"""PASS""",0.189,"""point_intersection""","""2026-11-07T18:39:17.784055+00:…",""""""
"""buildings""","""ok""",1,"""PASS""",0.194,"""point_intersection""","""2026-11-07T18:39:19.940667+00:…",""""""
"""lber""","""ok""",5,"""PASS""",0.191,"""within_250m_candidate""","""2026-11-07T18:39:21.882940+00:…",""""""
"""tariffs""","""ok""",2,"""PASS""",0.377,"""provider_location_candidates""","""2026-11-07T18:39:22.416739+00:…",""""""
"""solar""","""ok""",5,"""PASS""",0.005,"""municipality_only""","""2026-11-07T17:25:43.566150+00:…",""""""
"""flood""","""ok""",1,"""PASS""",0.816,"""point_intersection""","""2026-11-07T18:39:25.275917+00:…",""""""
"""elevation""","""ok""",1,"""PASS""",0.076,"""coordinate_sample""","""2026-11-07T18:39:25.531140+00:…",""""""


HTTP checks: 19/19 PASS.
Individual provider checks: 8/8 PASS, 0 FAIL.
Combined lookup provider errors: []


To reuse the backend directly in Python without an HTTP server:

```python
from API import lookup_site
site = lookup_site(LATITUDE, LONGITUDE)
```

For an API caller, the minimal request is:

```python
site = requests.get(
    f"{BASE_URL}/site",
    params={"latitude": LATITUDE, "longitude": LONGITUDE},
    timeout=120,
).json()
```

Use `refresh=true` only when you need a new MassCEC download. Cached solar data keeps its original download timestamp. Other providers are queried live on each request.